In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

# Sklearn
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import joblib

print("✅ All libraries imported successfully")

Matplotlib is building the font cache; this may take a moment.


✅ All libraries imported successfully


In [2]:
# ===============================
# Cell 2: Paths and Config
# ===============================
DATASET_PATH = "../dataset/PlantVillage"
ML_MODEL_PATH = "../models/ml"
DL_MODEL_PATH = "../models/dl"
RESULTS_PATH = "../results"

# Create folders if not exist
os.makedirs(ML_MODEL_PATH, exist_ok=True)
os.makedirs(DL_MODEL_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

# Config
IMAGE_SIZE = 224
BATCH_SIZE = 32
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", DEVICE)

✅ Using device: cpu


In [5]:
# ===============================
# Cell 3: Dataset Transform + Load
# ===============================
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)

class_names = dataset.classes
num_classes = len(class_names)

print(f"✅ Total images: {len(dataset)}")
print(f"✅ Total classes: {num_classes}")
print("✅ Classes:", class_names[:5], "...")

✅ Total images: 20638
✅ Total classes: 15
✅ Classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy'] ...


In [6]:
# ===============================
# Cell 4: Train / Val / Test Split
# ===============================
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("✅ Dataset split completed")
print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

✅ Dataset split completed
Train: 14446
Val: 3095
Test: 3097


In [7]:
# ===============================
# Cell 5: Load Pretrained ResNet as Feature Extractor
# ===============================
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Remove final classification layer
feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()

print("✅ ResNet50 feature extractor loaded")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\aminr/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [01:31<00:00, 1.13MB/s]


✅ ResNet50 feature extractor loaded


In [8]:
# ===============================
# Cell 6: Feature Extraction Function
# ===============================
def extract_features(dataloader, model, device):
    features = []
    labels = []

    with torch.no_grad():
        for images, targets in tqdm(dataloader):
            images = images.to(device)

            outputs = model(images)
            outputs = outputs.view(outputs.size(0), -1)

            features.append(outputs.cpu().numpy())
            labels.append(targets.numpy())

    features = np.vstack(features)
    labels = np.hstack(labels)

    return features, labels

print("✅ Feature extraction function ready")

✅ Feature extraction function ready


In [9]:
# ===============================
# Cell 7: Extract Train/Test Features
# ===============================
X_train, y_train = extract_features(train_loader, feature_extractor, DEVICE)
X_test, y_test = extract_features(test_loader, feature_extractor, DEVICE)

print("✅ Train features shape:", X_train.shape)
print("✅ Test features shape:", X_test.shape)

100%|██████████| 97/97 [10:31<00:00,  6.51s/it]

✅ Train features shape: (14446, 2048)
✅ Test features shape: (3097, 2048)


In [10]:
# ===============================
# Cell 8: Feature Scaling
# ===============================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(scaler, f"{ML_MODEL_PATH}/scaler.pkl")

print("✅ Features scaled and scaler saved")

✅ Features scaled and scaler saved


In [11]:
# ===============================
# Cell 9: Train SVM
# ===============================
svm_model = SVC(kernel="rbf", C=10, gamma="scale")
svm_model.fit(X_train, y_train)

svm_preds = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)

print(f"✅ SVM Accuracy: {svm_acc * 100:.2f}%")

joblib.dump(svm_model, f"{ML_MODEL_PATH}/svm_resnet.pkl")

✅ SVM Accuracy: 91.93%


['../models/ml/svm_resnet.pkl']

In [12]:
# ===============================
# Cell 10: Train Random Forest
# ===============================
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=SEED,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)

print(f"✅ Random Forest Accuracy: {rf_acc * 100:.2f}%")

joblib.dump(rf_model, f"{ML_MODEL_PATH}/rf_resnet.pkl")

✅ Random Forest Accuracy: 83.40%


['../models/ml/rf_resnet.pkl']

In [13]:
# ===============================
# Cell 11: Train k-NN
# ===============================
knn_model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_model.fit(X_train, y_train)

knn_preds = knn_model.predict(X_test)
knn_acc = accuracy_score(y_test, knn_preds)

print(f"✅ k-NN Accuracy: {knn_acc * 100:.2f}%")

joblib.dump(knn_model, f"{ML_MODEL_PATH}/knn_resnet.pkl")

✅ k-NN Accuracy: 78.66%


['../models/ml/knn_resnet.pkl']

In [14]:
# ===============================
# Cell 12: Compare ML Results
# ===============================
ml_results = pd.DataFrame({
    "Model": ["SVM", "Random Forest", "k-NN"],
    "Accuracy": [svm_acc, rf_acc, knn_acc]
})

ml_results["Accuracy (%)"] = ml_results["Accuracy"] * 100
ml_results = ml_results.sort_values("Accuracy (%)", ascending=False)

print("📊 ML Model Comparison")
display(ml_results)

ml_results.to_csv(f"{RESULTS_PATH}/ml_results.csv", index=False)

📊 ML Model Comparison


,Model,Accuracy,Accuracy (%)
0,SVM,0.919277,91.927672
1,Random Forest,0.834033,83.403294
2,k-NN,0.786568,78.656765


In [15]:
# ===============================
# Cell 13: Training + Evaluation Functions
# ===============================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    loss = running_loss / len(loader)

    return loss, acc


def evaluate_model(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

print("✅ Training utilities ready")

✅ Training utilities ready


In [16]:
# ===============================
# Cell 14: Load AlexNet
# ===============================
alexnet = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)

# Replace final layer
alexnet.classifier[6] = nn.Linear(alexnet.classifier[6].in_features, num_classes)

alexnet = alexnet.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(alexnet.parameters(), lr=0.0001)

print("✅ AlexNet ready")

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to C:\Users\aminr/.cache\torch\hub\checkpoints\alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [07:00<00:00, 581kB/s]    


✅ AlexNet ready


In [17]:
# ===============================
# Cell 15: Train AlexNet
# ===============================
EPOCHS = 5
alexnet_history = []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        alexnet, train_loader, criterion, optimizer, DEVICE
    )

    val_acc = evaluate_model(alexnet, val_loader, DEVICE)

    alexnet_history.append([epoch + 1, train_loss, train_acc, val_acc])

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

torch.save(alexnet.state_dict(), f"{DL_MODEL_PATH}/alexnet.pth")
print("✅ AlexNet model saved")

100%|██████████| 452/452 [20:52<00:00,  2.77s/it]


Epoch 1/5 | Loss: 0.4650 | Train Acc: 84.85% | Val Acc: 88.01%


100%|██████████| 452/452 [25:00<00:00,  3.32s/it]


Epoch 2/5 | Loss: 0.1456 | Train Acc: 95.26% | Val Acc: 96.54%


100%|██████████| 452/452 [33:17<00:00,  4.42s/it]


Epoch 3/5 | Loss: 0.0828 | Train Acc: 97.23% | Val Acc: 96.22%


100%|██████████| 452/452 [26:25<00:00,  3.51s/it]


Epoch 4/5 | Loss: 0.0695 | Train Acc: 97.76% | Val Acc: 97.06%


100%|██████████| 452/452 [11:55:38<00:00, 95.00s/it]       


Epoch 5/5 | Loss: 0.0415 | Train Acc: 98.64% | Val Acc: 98.16%
✅ AlexNet model saved


In [ ]:
# ===============================
# Cell 16: Load ResNet for Training
# ===============================
resnet_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze all layers
for param in resnet_model.parameters():
    param.requires_grad = False

# Replace final layer
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

resnet_model = resnet_model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet_model.fc.parameters(), lr=0.0001)

print("✅ ResNet ready (fine-tuning mode)")

✅ ResNet ready (fine-tuning mode)


In [19]:
# ===============================
# Cell 17: Train ResNet (FAST VERSION)
# ===============================
EPOCHS = 3   # Keep low for CPU

resnet_history = []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        resnet_model, train_loader, criterion, optimizer, DEVICE
    )

    val_acc = evaluate_model(resnet_model, val_loader, DEVICE)

    resnet_history.append([epoch + 1, train_loss, train_acc, val_acc])

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

torch.save(resnet_model.state_dict(), f"{DL_MODEL_PATH}/resnet.pth")
print("✅ ResNet model saved")

100%|██████████| 452/452 [1:48:09<00:00, 14.36s/it]    


Epoch 1/3 | Loss: 2.0548 | Train Acc: 47.42% | Val Acc: 70.60%


100%|██████████| 452/452 [1:01:23<00:00,  8.15s/it]


Epoch 2/3 | Loss: 1.3906 | Train Acc: 73.78% | Val Acc: 79.22%


100%|██████████| 452/452 [3:43:53<00:00, 29.72s/it]     


Epoch 3/3 | Loss: 1.0646 | Train Acc: 80.47% | Val Acc: 82.36%
✅ ResNet model saved


In [20]:
# ===============================
# Cell 18: Evaluate ResNet on Test Set
# ===============================
test_acc = evaluate_model(resnet_model, test_loader, DEVICE)

print(f"🔥 ResNet Test Accuracy: {test_acc * 100:.2f}%")

🔥 ResNet Test Accuracy: 83.40%


In [21]:
# ===============================
# Cell 19: Load MobileNet
# ===============================
mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze feature extractor
for param in mobilenet.features.parameters():
    param.requires_grad = False

# Replace classifier
mobilenet.classifier[1] = nn.Linear(
    mobilenet.classifier[1].in_features,
    num_classes
)

mobilenet = mobilenet.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    mobilenet.classifier.parameters(),
    lr=0.0001
)

print("✅ MobileNet ready")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\aminr/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:06<00:00, 2.36MB/s]


✅ MobileNet ready


In [22]:
# ===============================
# Cell 20: Train MobileNet
# ===============================
EPOCHS = 3

mobilenet_history = []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        mobilenet, train_loader, criterion, optimizer, DEVICE
    )

    val_acc = evaluate_model(mobilenet, val_loader, DEVICE)

    mobilenet_history.append([epoch + 1, train_loss, train_acc, val_acc])

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

torch.save(mobilenet.state_dict(), f"{DL_MODEL_PATH}/mobilenet.pth")
print("✅ MobileNet model saved")

100%|██████████| 452/452 [21:11<00:00,  2.81s/it]


Epoch 1/3 | Loss: 2.0281 | Train Acc: 48.33% | Val Acc: 72.21%


100%|██████████| 452/452 [20:16<00:00,  2.69s/it]


Epoch 2/3 | Loss: 1.3346 | Train Acc: 74.81% | Val Acc: 81.20%


100%|██████████| 452/452 [20:45<00:00,  2.75s/it]


Epoch 3/3 | Loss: 1.0053 | Train Acc: 80.69% | Val Acc: 84.26%
✅ MobileNet model saved


In [23]:
# ===============================
# Cell 21: Evaluate MobileNet
# ===============================
mobilenet_test_acc = evaluate_model(mobilenet, test_loader, DEVICE)

print(f"⚡ MobileNet Test Accuracy: {mobilenet_test_acc * 100:.2f}%")

⚡ MobileNet Test Accuracy: 84.11%
